In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

UNIT_MODE = "auto"  # "auto" | "seconds" | "minutes"
SEC_TO_MIN_THRESHOLD = 600.0  # if Δt95% > 600 (≈10 min in seconds), assume seconds and /60

ROOT = "rt_pred_unmodified/exp_hela/"  # Path location of dataset
FILES = ["hela_unmodified.txt"]

# Default model knobs (used if HP_SEARCH = False)
EPOCHS = 500
BATCH = 256
D_MODEL = 256
N_LAYERS = 16
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.10
CONV_K = 9
HUBER_DELTA = 1.0
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 4000
MIN_LR = 1e-5
BASE_LR = 2e-3

# Hyperparameter search config
HP_SEARCH = True          # Turn OFF while using default config
EPOCHS_TUNE = 500          # Max epochs per HP trial (with early stopping)

HP_CONFIGS = [
    {
        "name": "small_d192_l8",
        "D_MODEL": 192,
        "N_LAYERS": 8,
        "N_HEADS": 4,
        "D_FF": 768,
        "DROPOUT": 0.10,
        "BASE_LR": 2e-3,
    },
    {
        "name": "baseline_d256_l8",
        "D_MODEL": 256,
        "N_LAYERS": 8,
        "N_HEADS": 8,
        "D_FF": 1024,
        "DROPOUT": 0.10,
        "BASE_LR": 2e-3,
    },
    {
        "name": "deep_d256_l12",
        "D_MODEL": 256,
        "N_LAYERS": 12,
        "N_HEADS": 8,
        "D_FF": 1024,
        "DROPOUT": 0.15,
        "BASE_LR": 1.5e-3,
    },
    {
        "name": "wide_d320_l12",
        "D_MODEL": 320,
        "N_LAYERS": 12,
        "N_HEADS": 8,
        "D_FF": 1280,
        "DROPOUT": 0.15,
        "BASE_LR": 1.5e-3,
    },
]

# ─────────────────────────────────────────────────────────────────────────────
# GPU + Mixed precision
# ─────────────────────────────────────────────────────────────────────────────

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

print("GPUs:", tf.config.list_physical_devices("GPU"))

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision:", mixed_precision.global_policy().name)

def set_seed(seed=42):
    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# Tokenizer
# ─────────────────────────────────────────────────────────────────────────────

PROSIT_CORE = "ACDEFGHIKLMNPQRSTVWY"
PROSIT_OX = "ACDEFGHIKLMNPQRSTVWYo"
DP_ALPHABET = "ACDEFGHIKLMNPQRSTVWY1234*"
PAD = 0

def infer_alphabet(seqs):
    if any(any(c in s for c in "1234*") for s in seqs):
        return DP_ALPHABET
    if any("o" in s for s in seqs):
        return PROSIT_OX
    return PROSIT_CORE

def build_tokenizer(alphabet):
    t = {c: i + 1 for i, c in enumerate(alphabet)}
    t["[CLS]"] = len(alphabet) + 1
    return t

def encode_sequence(seq, tok, max_len):
    ids = [tok["[CLS]"]] + [tok[c] for c in seq if c in tok]
    if len(ids) > max_len + 1:
        ids = ids[: max_len + 1]
    return ids + [PAD] * ((max_len + 1) - len(ids))

# ─────────────────────────────────────────────────────────────────────────────
# LR Schedule / Optimizer
# ─────────────────────────────────────────────────────────────────────────────

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, min_lr=1e-5, total_steps=200_000):
        super().__init__()
        self.base_lr = tf.cast(base_lr, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.min_lr = tf.cast(min_lr, tf.float32)
        self.total = tf.cast(total_steps, tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        # Warmup phase
        warm = self.base_lr * tf.minimum(1.0, step / tf.maximum(1.0, self.warmup_steps))

        # Cosine decay phase
        progress = tf.clip_by_value(
            (step - self.warmup_steps) / tf.maximum(1.0, self.total - self.warmup_steps),
            0.0,
            1.0,
        )
        cosine = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
            1.0 + tf.cos(np.pi * progress)
        )

        return tf.where(step < self.warmup_steps, warm, cosine)


# ─────────────────────────────────────────────────────────────────────────────
# Layers
# ─────────────────────────────────────────────────────────────────────────────

class StripMask(tf.keras.layers.Layer):
    """Stops Keras mask propagation to silence harmless warnings."""
    def __init__(self):
        super().__init__()
        self.supports_masking = True

    def call(self, x):
        return x

    def compute_mask(self, inputs, mask=None):
        return None


class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len_with_cls, d_model, dropout):
        super().__init__()
        self.supports_masking = True
        self.pos = tf.keras.layers.Embedding(
            max_len_with_cls, d_model, name="positional_embedding"
        )
        self.do = tf.keras.layers.Dropout(dropout)

    def call(self, token_emb, training=None):
        L = tf.shape(token_emb)[1]
        pos_ids = tf.range(L)
        x = token_emb + self.pos(pos_ids)[None, ...]
        return self.do(x, training=training)

    def compute_mask(self, inputs, mask=None):
        return mask

class ConvModule(tf.keras.layers.Layer):
    """Pointwise (GLU) → DepthwiseConv1D → BN → SiLU → Pointwise."""
    def __init__(self, d_model, kernel_size=9, dropout=0.1):
        super().__init__()
        self.pw1 = tf.keras.layers.Dense(2 * d_model)  # GLU gating
        self.dw = tf.keras.layers.DepthwiseConv1D(kernel_size, padding="same")
        self.bn = tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
        self.act = tf.keras.layers.Activation(tf.nn.silu)
        self.pw2 = tf.keras.layers.Dense(d_model)
        self.do = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=None):
        u, g = tf.split(self.pw1(x), 2, axis=-1)
        x = u * tf.keras.activations.sigmoid(g)
        x = self.dw(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.pw2(x)
        return self.do(x, training=training)


class GEGLUFFN(tf.keras.layers.Layer):
    """GEGLU feed-forward: Dense(2*d_ff) -> GEGLU -> Dropout -> Dense(d_model)."""
    def __init__(self, d_ff, d_model, dropout):
        super().__init__()
        self.pre = tf.keras.layers.Dense(2 * d_ff)
        self.do = tf.keras.layers.Dropout(dropout)
        self.proj = tf.keras.layers.Dense(d_model)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=None):
        x = self.norm(x)
        h = self.pre(x)
        a, b = tf.split(h, 2, axis=-1)
        geglu = tf.keras.activations.gelu(a) * b
        geglu = self.do(geglu, training=training)
        return self.proj(geglu)

class EncoderBlock(tf.keras.layers.Layer):
    """
    Conformer-style macaron block:
      0.5*FFN1 -> MHSA -> ConvModule -> 0.5*FFN2
    """
    def __init__(self, d_model, n_heads, d_ff, dropout, conv_k):
        super().__init__()

        # Macaron FFNs (each has its own LayerNorm inside GEGLUFFN)
        self.ffn1 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)
        self.ffn2 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)

        # MHSA
        self.norm_attn = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=n_heads,
            key_dim=d_model // n_heads,
            dropout=dropout,
        )
        self.do_attn = tf.keras.layers.Dropout(dropout)

        # Conv module
        self.conv_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.conv = ConvModule(d_model, kernel_size=conv_k, dropout=dropout)

    def call(self, x, attn_mask, training=None):
        # Macaron FFN1 (pre-norm inside GEGLUFFN) with 0.5 residual
        x = x + 0.5 * self.ffn1(x, training=training)

        # MHSA (pre-norm)
        y = self.mha(
            self.norm_attn(x),
            self.norm_attn(x),
            attention_mask=attn_mask,
            training=training,
        )
        x = x + self.do_attn(y, training=training)

        # Conv module (pre-norm)
        y = self.conv(self.conv_norm(x), training=training)
        x = x + y

        # Macaron FFN2 (pre-norm inside GEGLUFFN) with 0.5 residual
        x = x + 0.5 * self.ffn2(x, training=training)

        return x


class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(
        self,
        vocab_size,
        max_len_with_cls,
        d_model,
        d_ff,
        n_layers,
        n_heads,
        dropout,
        conv_k,
    ):
        super().__init__()
        self.embed = tf.keras.layers.Embedding(
            vocab_size,
            d_model,
            mask_zero=True,
            name="aa_embedding",
        )
        self.pos = PositionalEmbedding(max_len_with_cls, d_model, dropout)
        self.strip = StripMask()
        self.blocks = [
            EncoderBlock(d_model, n_heads, d_ff, dropout, conv_k)
            for _ in range(n_layers)
        ]
        self.final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, token_ids, training=None):
        key_padding_mask = tf.not_equal(token_ids, 0)  # [B, L] bool

        x = self.embed(token_ids)  # [B, L, D]
        x = self.pos(x, training=training)
        x = self.strip(x)  # stop mask propagation to silence warnings

        attn_mask = tf.cast(key_padding_mask[:, None, :], tf.bool)  # [B, 1, L]

        for blk in self.blocks:
            x = blk(x, attn_mask, training=training)

        return self.final_norm(x), key_padding_mask


class MaskedMeanMax(tf.keras.layers.Layer):
    """Pools [B,L,D] with mask [B,L] → returns [mean, max], each [B,D]."""
    def call(self, inputs):
        x, mask = inputs
        mask = tf.cast(mask, x.dtype)[:, :, None]  # [B, L, 1]

        # Mean
        sum_x = tf.reduce_sum(x * mask, axis=1)  # [B, D]
        length = tf.reduce_sum(mask, axis=1)  # [B, 1] for safe broadcast
        length = tf.maximum(length, tf.constant(1.0, x.dtype))
        mean = sum_x / length  # [B, D]

        # Max (pad positions to very negative)
        very_neg = tf.cast(-1e4, x.dtype)
        x_masked = tf.where(tf.cast(mask, tf.bool), x, very_neg)
        maxp = tf.reduce_max(x_masked, axis=1)  # [B, D]

        return [mean, maxp]

class AttnPool(tf.keras.layers.Layer):
    """
    Single-head learned attention pooling over sequence.
    Input:  x [B, L, D], mask [B, L] (bool)
    Output: pooled [B, D]
    """
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.query = self.add_weight(
            name="attn_query",
            shape=(1, 1, d_model),
            initializer="glorot_uniform",
            trainable=True,
        )

    def call(self, inputs):
        x, mask = inputs   # x: [B,L,D], mask: [B,L] bool
        B = tf.shape(x)[0]
        D = tf.shape(x)[-1]

        q = tf.cast(self.query, x.dtype)      # [1,1,D]
        q = tf.tile(q, [B, 1, 1])             # [B,1,D]

        # scores: [B,1,L]
        scale = tf.math.sqrt(tf.cast(D, x.dtype))
        scores = tf.matmul(q, x, transpose_b=True) / scale

        # mask: 1 for valid, 0 for pad
        mask_f = tf.cast(mask[:, None, :], x.dtype)   # [B,1,L]
        scores = scores + (1.0 - mask_f) * tf.cast(-1e4, x.dtype)

        attn = tf.nn.softmax(scores, axis=-1)         # [B,1,L]
        ctx = tf.matmul(attn, x)                      # [B,1,D]
        return ctx[:, 0, :]                           # [B,D]


# ─────────────────────────────────────────────────────────────────────────────
# Utilities
# ─────────────────────────────────────────────────────────────────────────────

def load_tsv(path):
    df = pd.read_csv(path, sep=None, engine="python")
    cols = {c.lower(): c for c in df.columns}
    seq_col = cols.get("sequence")
    rt_col = cols.get("rt")

    if seq_col is None or rt_col is None:
        raise ValueError(
            f"sequence/rt columns not found in {path}. Columns: {list(df.columns)}"
        )

    df = df[[seq_col, rt_col]].rename(columns={seq_col: "sequence", rt_col: "rt"})
    df["sequence"] = (
        df["sequence"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )
    df["rt"] = pd.to_numeric(df["rt"], errors="coerce")
    df = df.dropna().reset_index(drop=True)
    return df

def make_ds(X, y=None, batch=BATCH, shuffle=False):
    ds = (
        tf.data.Dataset.from_tensor_slices((X, y))
        if y is not None
        else tf.data.Dataset.from_tensor_slices(X)
    )
    if shuffle:
        ds = ds.shuffle(len(X), seed=42)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def pearson_r(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.size < 2:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def p95_width(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.percentile(x, 97.5) - np.percentile(x, 2.5))

def residual_ci95(residuals):
    """95% confidence interval (2.5–97.5 percentile) of residuals in minutes."""
    r = np.asarray(residuals, dtype=np.float64)
    lo = float(np.percentile(r, 2.5))
    hi = float(np.percentile(r, 97.5))
    return lo, hi

def normalize_rt_units(y_raw, unit_mode="auto"):
    y = np.asarray(y_raw, dtype=np.float64)
    dt95 = p95_width(y)

    if unit_mode == "minutes":
        return y, "minutes"
    if unit_mode == "seconds":
        return y / 60.0, "seconds→minutes(/60)"

    if dt95 > SEC_TO_MIN_THRESHOLD:
        return y / 60.0, "seconds→minutes(/60)"
    else:
        return y, "minutes"


# ─────────────────────────────────────────────────────────────────────────────
# Model factory + Hyperparameter tuning
# ─────────────────────────────────────────────────────────────────────────────

def build_model_from_hp(hp, max_len_with_cls, vocab_size, steps_per_epoch, epochs):
    """
    Build a Conformer-lite model using a hyperparameter dict `hp`.
    """
    d_model = hp["D_MODEL"]
    n_layers = hp["N_LAYERS"]
    n_heads = hp["N_HEADS"]
    d_ff = hp["D_FF"]
    dropout = hp.get("DROPOUT", DROPOUT)
    base_lr = hp.get("BASE_LR", BASE_LR)

    inp = tf.keras.Input(
        shape=(max_len_with_cls,),
        dtype=tf.int32,
        name="tokens",
    )

    enc = TransformerEncoder(
        vocab_size=vocab_size,
        max_len_with_cls=max_len_with_cls,
        d_model=d_model,
        d_ff=d_ff,
        n_layers=n_layers,
        n_heads=n_heads,
        dropout=dropout,
        conv_k=CONV_K,
    )

    x, key_mask = enc(inp)  # x: [B, L, D], key_mask: [B, L]

    cls_tok = x[:, 0, :]  # [B, D]
    mean_p, max_p = MaskedMeanMax()([x, key_mask])  # [B, D], [B, D]

    attn_pool = AttnPool(d_model=d_model, name="attn_pool")
    attn_vec = attn_pool([x, key_mask])  # [B, D]

    # Combine four summaries: CLS, mean, max, attention-pooled
    feat = tf.keras.layers.Concatenate()(
        [cls_tok, mean_p, max_p, attn_vec]
    )  # [B, 4D]

    feat = tf.keras.layers.LayerNormalization(epsilon=1e-6)(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)
    feat = tf.keras.layers.Dense(
        d_model * 2,
        activation=tf.keras.activations.gelu,
    )(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)

    out = tf.keras.layers.Dense(
        1,
        activation="linear",
        dtype="float32",   # Regression head in float32
    )(feat)

    model = tf.keras.Model(inp, out, name=f"rt_conformer_lite_{hp.get('name','hp')}")

    total_steps = max(1, steps_per_epoch * epochs)
    lr_sched = WarmupCosine(
        base_lr,
        warmup_steps=WARMUP_STEPS,
        min_lr=MIN_LR,
        total_steps=total_steps,
    )

    opt = tf.keras.optimizers.AdamW(
        learning_rate=lr_sched,
        weight_decay=WEIGHT_DECAY,
        epsilon=1e-8,
        global_clipnorm=1.0,
    )

    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA),
    )

    return model


def tune_hyperparams(X_tr, y_tr, X_val, y_val, max_len_with_cls, vocab_size):
    """
    Simple manual hyperparameter search over HP_CONFIGS.
    Returns the best hp dict.
    """
    print(f"\n[HP SEARCH] Train size: {len(X_tr)}, Val size: {len(X_val)}")
    steps_per_epoch = max(1, len(X_tr) // BATCH)

    best_hp = None
    best_loss = np.inf

    for i, hp in enumerate(HP_CONFIGS):
        print(f"\n[HP {i+1}/{len(HP_CONFIGS)}] {hp['name']}")
        print("  config:", {k: v for k, v in hp.items() if k != "name"})

        model = build_model_from_hp(
            hp,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
            steps_per_epoch=steps_per_epoch,
            epochs=EPOCHS_TUNE,
        )

        ds_tr = make_ds(X_tr, y_tr[:, None], batch=BATCH, shuffle=True)
        ds_val = make_ds(X_val, y_val[:, None], batch=BATCH)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=100,
                restore_best_weights=True,
                verbose=1,
            )
        ]

        hist = model.fit(
            ds_tr,
            validation_data=ds_val,
            epochs=EPOCHS_TUNE,
            verbose=2,
            callbacks=callbacks,
        )

        val_loss = float(min(hist.history["val_loss"]))
        print(f"  best val_loss for {hp['name']}: {val_loss:.6f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_hp = hp

    print(f"\n[HP SEARCH] Best config: {best_hp['name']} (val_loss={best_loss:.6f})")
    return best_hp


# ─────────────────────────────────────────────────────────────────────────────
# 5-folds cross-validation
# ─────────────────────────────────────────────────────────────────────────────

def run_cross_validation(
    X,
    y_scaled,
    df,
    y_mean,
    y_std,
    best_hp,
    max_len_with_cls,
    vocab_size,
    file_stem,
):
    """
    5-fold cross-validation over the entire dataset using best_hp.
    Saves:
      - {stem}_cv_metrics.csv
      - {stem}_test_predictions_cv.csv
    """
    print("\n[CV] Running 5-fold cross validation...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    metrics_rows = []
    preds_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"\n[CV] Fold {fold}/5")
        X_train_f = X[train_idx]
        X_val_f = X[val_idx]
        y_train_f = y_scaled[train_idx]
        y_val_f = y_scaled[val_idx]

        ds_train_f = make_ds(X_train_f, y_train_f[:, None], batch=BATCH, shuffle=True)
        ds_val_f = make_ds(X_val_f, y_val_f[:, None], batch=BATCH)

        steps_per_epoch = max(1, len(X_train_f) // BATCH)
        model = build_model_from_hp(
            best_hp,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
            steps_per_epoch=steps_per_epoch,
            epochs=EPOCHS,
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=100,
                restore_best_weights=True,
                verbose=1,
            )
        ]

        model.fit(
            ds_train_f,
            validation_data=ds_val_f,
            epochs=EPOCHS,
            verbose=2,
            callbacks=callbacks,
        )

        # Predictions on validation fold
        ds_val_tokens = make_ds(X_val_f, batch=BATCH, shuffle=False)
        y_val_pred_scaled = model.predict(ds_val_tokens, verbose=0).reshape(-1)

        y_true = y_val_f * y_std + y_mean
        y_pred = y_val_pred_scaled * y_std + y_mean

        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        R = pearson_r(y_true, y_pred)
        delta_t95 = p95_width(y_true)
        residuals = y_pred - y_true
        delta_tr95 = p95_width(residuals)
        delta_tr95_pct = (
            float(100.0 * delta_tr95 / delta_t95) if delta_t95 > 0 else np.nan
        )
        ci_low, ci_high = residual_ci95(residuals)

        metrics_rows.append(
            dict(
                file=file_stem,
                fold=fold,
                n=len(val_idx),
                R=R,
                R2=r2,
                Dt95_min=delta_t95,
                Dtr95_pct=delta_tr95_pct,
                MAE_min=mae,
                MSE_min2=mse,
                RMSE_min=rmse,
                CI_low_min=ci_low,
                CI_high_min=ci_high,
            )
        )

        # Save predictions for this fold
        for local_i, global_idx in enumerate(val_idx):
            preds_rows.append(
                dict(
                    file=file_stem,
                    fold=fold,
                    index=int(global_idx),
                    sequence=df.iloc[global_idx]["sequence"],
                    rt_true_min=float(y_true[local_i]),
                    rt_pred_min=float(y_pred[local_i]),
                )
            )

    metrics_df = pd.DataFrame(metrics_rows).rename(
        columns={
            "R2": "R²",
            "Dt95_min": "Δt₉₅% (min)",
            "Dtr95_pct": "Δtr₉₅%",
            "MAE_min": "MAE (min)",
            "MSE_min2": "MSE (min²)",
            "RMSE_min": "RMSE (min)",
            "CI_low_min": "CI_low (min)",
            "CI_high_min": "CI_high (min)",
        }
    )

    cv_metrics_path = f"rt_pred_unmodified/exp_hela/{file_stem}_cv_metrics.csv"
    metrics_df.to_csv(cv_metrics_path, index=False)
    print("[CV] Saved metrics →", cv_metrics_path)

    preds_df = pd.DataFrame(preds_rows).sort_values(["fold", "index"])
    cv_preds_path = f"rt_pred_unmodified/exp_hela/{file_stem}_test_predictions_cv.csv"
    preds_df.to_csv(cv_preds_path, index=False)
    print("[CV] Saved predictions →", cv_preds_path)


# ─────────────────────────────────────────────────────────────────────────────
# Train & Evaluate one file
# ─────────────────────────────────────────────────────────────────────────────

def train_one_file(path):
    file_name = os.path.basename(path)
    file_stem = os.path.splitext(file_name)[0]

    print(f"\n=== {file_name} ===")
    df = load_tsv(path)
    seqs = df["sequence"].tolist()

    # Units → minutes
    y_minutes, detected = normalize_rt_units(df["rt"].values, unit_mode=UNIT_MODE)
    print(f"[Unit] Detected/used units for metrics: {detected}")

    # z-score target for training
    y_mean = float(y_minutes.mean())
    y_std_raw = y_minutes.std()
    y_std = float(y_std_raw if y_std_raw > 1e-6 else 1.0)
    y_scaled = ((y_minutes - y_mean) / y_std).astype(np.float32)
    alphabet = infer_alphabet(seqs)
    tok = build_tokenizer(alphabet)
    max_len = max(len(s) for s in seqs)
    max_len_with_cls = max_len + 1
    vocab_size = max(tok.values()) + 1

    X = np.stack([encode_sequence(s, tok, max_len) for s in seqs]).astype(np.int32)
    indices = np.arange(len(df))

    # Train/test split
    idx_tr, idx_te, X_tr, X_te, y_tr, y_te = train_test_split(
        indices,
        X,
        y_scaled,
        test_size=0.20,
        random_state=42,
    )

    # Validation split from training data during Hyperparameter tuning
    if HP_SEARCH:
        X_tr_sub, X_val, y_tr_sub, y_val = train_test_split(
            X_tr, y_tr, test_size=0.20, random_state=123
        )
        best_hp = tune_hyperparams(
            X_tr_sub,
            y_tr_sub,
            X_val,
            y_val,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
        )
    else:
        best_hp = {
            "name": "manual_default",
            "D_MODEL": D_MODEL,
            "N_LAYERS": N_LAYERS,
            "N_HEADS": N_HEADS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
            "BASE_LR": BASE_LR,
        }
        print("\n[HP SEARCH] Disabled, using manual defaults:", best_hp)

    # 5-fold CV on full dataset (using best_hp)
    run_cross_validation(
        X,
        y_scaled,
        df,
        y_mean,
        y_std,
        best_hp,
        max_len_with_cls=max_len_with_cls,
        vocab_size=vocab_size,
        file_stem=file_stem,
    )

    # Final training on full training set with best hyperparameters
    steps_per_epoch = max(1, len(X_tr) // BATCH)
    model = build_model_from_hp(
        best_hp,
        max_len_with_cls=max_len_with_cls,
        vocab_size=vocab_size,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS,
    )

    ds_tr_full = make_ds(X_tr, y_tr[:, None], batch=BATCH, shuffle=True)
    ds_va = make_ds(X_te, y_te[:, None], batch=BATCH)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=100,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    print("\n[TRAIN] Final model training with best hyperparameters...")
    history = model.fit(
        ds_tr_full,
        validation_data=ds_va,
        epochs=EPOCHS,
        verbose=2,
        callbacks=callbacks,
    )

    # Per-epoch training/validation loss save
    hist_df = pd.DataFrame(
        {
            "epoch": np.arange(1, len(history.history["loss"]) + 1),
            "loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }
    )
    hist_path = f"rt_pred_unmodified/exp_hela/{file_stem}_train_history.csv"
    hist_df.to_csv(hist_path, index=False)
    print("[TRAIN] Saved train history →", hist_path)

    # Predict & invert scaling back to minutes
    ds_tr_tokens_noshuf = make_ds(X_tr, batch=BATCH, shuffle=False)
    y_tr_pred_scaled = model.predict(ds_tr_tokens_noshuf, verbose=0).reshape(-1)
    y_tr_true = y_tr * y_std + y_mean
    y_tr_pred = y_tr_pred_scaled * y_std + y_mean
    ds_te_tokens = make_ds(X_te, batch=BATCH, shuffle=False)
    y_pred_scaled = model.predict(ds_te_tokens, verbose=0).reshape(-1)
    y_true = y_te * y_std + y_mean
    y_pred = y_pred_scaled * y_std + y_mean

    # Train / Validation prediction CSV
    train_pred_df = pd.DataFrame(
        {
            "index": idx_tr,
            "sequence": df.iloc[idx_tr]["sequence"].values,
            "rt_true_min": y_tr_true.astype(float),
            "rt_pred_min": y_tr_pred.astype(float),
        }
    )
    train_pred_path = f"rt_pred_unmodified/exp_hela/{file_stem}_train_predictions.csv"
    train_pred_df.to_csv(train_pred_path, index=False)
    print("[PRED] Saved train predictions →", train_pred_path)

    val_pred_df = pd.DataFrame(
        {
            "index": idx_te,
            "sequence": df.iloc[idx_te]["sequence"].values,
            "rt_true_min": y_true.astype(float),
            "rt_pred_min": y_pred.astype(float),
        }
    )
    val_pred_path = f"rt_pred_unmodified/exp_hela/{file_stem}_validation_predictions.csv"
    val_pred_df.to_csv(val_pred_path, index=False)
    print("[PRED] Saved validation/test predictions →", val_pred_path)

    # Metrics (in minutes) for final model
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    R = pearson_r(y_true, y_pred)
    delta_t95 = p95_width(y_true)
    residuals = y_pred - y_true
    delta_tr95 = p95_width(residuals)
    delta_tr95_pct = (
        float(100.0 * delta_tr95 / delta_t95) if delta_t95 > 0 else np.nan
    )
    ci_low, ci_high = residual_ci95(residuals)

    print(f"Samples (n): {len(df)} | MaxLen: {max_len}")
    print(f"Best HP    : {best_hp['name']}")
    print(f"R        : {R:.6f}")
    print(f"R²       : {r2:.6f}")
    print(f"MAE      : {mae:.6f} min")
    print(f"MSE      : {mse:.6f} min^2")
    print(f"RMSE     : {rmse:.6f} min")
    print(f"Δt95%    : {delta_t95:.6f} min")
    print(
        f"Δtr95%   : {delta_tr95_pct:.3f} % "
        f"(absolute: {delta_tr95:.6f} min)"
    )
    print(
        f"95% CI (residuals, min): "
        f"[{ci_low:.6f}, {ci_high:.6f}]"
    )

    return dict(
        file=file_name,
        best_hp=best_hp["name"],
        n=len(df),
        max_len=max_len,
        R=R,
        R2=r2,
        Dt95_min=delta_t95,
        Dtr95_pct=delta_tr95_pct,
        MAE_min=mae,
        MSE_min2=mse,
        RMSE_min=rmse,
        CI_low_min=ci_low,
        CI_high_min=ci_high,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Run & Save
# ─────────────────────────────────────────────────────────────────────────────

results = []

for f in FILES:
    p = os.path.join(ROOT, f)
    if os.path.exists(p):
        results.append(train_one_file(p))
    else:
        print(f"Missing: {p}")

res_df = pd.DataFrame(results).rename(
    columns={
        "best_hp": "best_hp",
        "R2": "R²",
        "Dt95_min": "Δt₉₅% (min)",
        "Dtr95_pct": "Δtr₉₅%",
        "MAE_min": "MAE (min)",
        "MSE_min2": "MSE (min²)",
        "RMSE_min": "RMSE (min)",
        "CI_low_min": "CI_low (min)",
        "CI_high_min": "CI_high (min)",
    }
)

res_df = res_df[
    [
        "file",
        "best_hp",
        "n",
        "max_len",
        "R",
        "R²",
        "Δt₉₅% (min)",
        "Δtr₉₅%",
        "MAE (min)",
        "MSE (min²)",
        "RMSE (min)",
        "CI_low (min)",
        "CI_high (min)",
    ]
]

out_csv = "rt_pred_unmodified/exp_hela/rt_transformer_metrics.csv"
res_df.to_csv(out_csv, index=False)

print("\nSaved metrics →", out_csv)
print("─" * 80)
print(res_df.to_string(index=False))
print("─" * 80)